In [1]:
from PIL import Image
import os
import json
os.environ["DISPLAY"] = ":0"

import torch
from torch import nn
from torchvision import models, transforms
from executorch.runtime import Runtime

W0701 01:02:42.871000 13027 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
runtime = Runtime.get()

In [3]:
program = runtime.load_program("model.pte")
model = program.load_method("forward")

[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu0/regs/identification/midr_el1
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu1/regs/identification/midr_el1
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu2/regs/identification/midr_el1
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu3/regs/identification/midr_el1


In [2]:
torch.__version__

'2.12.1+cu130'

In [3]:
# The aarch64 version of pytorch requires using the qnnpack engine.
#torch.backends.quantized.engine = 'qnnpack'
torch.backends.quantized.engine = 'xnnpack'

RuntimeError: xnnpack is not a valid value for quantized engine

RuntimeError: xnnpack is not a valid value for quantized engine

In [5]:
!big-display off

In [ ]:
model = models.quantization.mobilenet_v2(pretrained=True, quantize=True)

/home/pi/Documents/notebooks/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/pi/Documents/notebooks/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_QuantizedWeights.IMAGENET1K_QNNPACK_V1`. You can also use `weights=MobileNet_V2_QuantizedWeights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
model = models.mobilenet_v2(pretrained=True)

/home/pi/Documents/notebooks/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/pi/Documents/notebooks/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [20]:
classes = []
with open("classes.json", "r") as f:
    classes = json.load(f)
    
#model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))

In [8]:
loaded_weights = torch.load("model.pt", weights_only=True)
model.load_state_dict({
    **model.state_dict(),
    "classifier.1.weight": loaded_weights["classifier.1.weight"],
    "classifier.1.bias": loaded_weights["classifier.1.bias"],
})

<All keys matched successfully>

In [9]:
model.qconfig = torch.quantization.get_default_qconfig('qnnpack')

In [10]:
quant_model = torch.quantization.prepare(model, inplace=False)

In [11]:
quant_model = torch.quantization.convert(quant_model, inplace=False)

/home/pi/Documents/notebooks/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:1272: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  warnings.warn(


In [12]:
quant_model = torch.jit.script(quant_model)

In [7]:
mobilenet_mean = [0.485, 0.456, 0.406]
mobilenet_std = [0.229, 0.224, 0.225]

validation_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])

unnormalize_transform = transforms.Normalize(
    mean=[-m / s for m, s in zip(mobilenet_mean, mobilenet_std)],
    std=[1 / s for s in mobilenet_std]
)

In [8]:
from picamera2 import Picamera2, Preview, Platform

In [9]:
cam = Picamera2()

[1:00:22.301685096] [11971]  INFO Camera camera_manager.cpp:340 libcamera v0.7.1+rpt20260609
[1:00:22.388642321] [12141]  INFO RPI pipeline_base.cpp:1133 Using configuration file '/usr/share/libcamera/pipeline/rpi/vc4/rpi_apps.yaml'
[1:00:22.654116913] [12141]  INFO IPAProxy ipa_proxy.cpp:184 Using tuning file /usr/share/libcamera/ipa/rpi/vc4/imx219.json
[1:00:22.668728843] [12141]  INFO Camera camera_manager.cpp:223 Adding camera '/base/soc/i2c0mux/i2c@1/imx219@10' for pipeline handler rpi/vc4
[1:00:22.668790065] [12141]  INFO RPI vc4.cpp:445 Registered camera /base/soc/i2c0mux/i2c@1/imx219@10 to Unicam device /dev/media2 and ISP device /dev/media0


In [47]:
config = cam.create_still_configuration(main={
    "size": (224, 224),
    "format": "BGR888",
}, display="main")
cam.configure(config)
cam.set_controls({"FrameRate": 36})
cam.start_preview(Preview.DRM)
cam.start()

[1:10:07.176487337] [12145]  INFO Camera camera.cpp:1216 configuring streams: (0) 224x224-BGR888/sRGB (1) 640x480-SBGGR10_CSI2P/RAW
[1:10:07.190361403] [12141]  INFO RPI vc4.cpp:620 Sensor: /base/soc/i2c0mux/i2c@1/imx219@10 - Selected sensor format: 640x480-SBGGR10_1X10/RAW - Selected unicam format: 640x480-pBAA/RAW


RuntimeError: An event loop is already running

In [45]:
# read frame
img = cam.capture_image("main")
input_tensor = validation_transforms(img)
input_batch = input_tensor.unsqueeze(0)
output = model.execute(input_batch)
print(classes[torch.tensor(output[0]).argmax().item()], output[0])

Cats tensor([[ 3.4321, -2.1822]])


/tmp/ipykernel_11971/839218654.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  print(classes[torch.tensor(output[0]).argmax().item()], output[0])


In [41]:
input_tensor = validation_transforms(img)

In [42]:
input_batch = input_tensor.unsqueeze(0)

In [ ]:
model.eval()
model(input_batch)

/tmp/ipykernel_11971/2049177532.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  classes[torch.tensor(output[0]).argmax().item()], output[0]


('Cats', tensor([[ 1.5431, -0.5612]]))

In [46]:
cam.stop()

In [21]:
classes

['Cats', 'Trains']